In [0]:
#Read Bronze
sales_df = spark.table("retail_catalog.retail_bronze.sales")
customers_df = spark.table("retail_catalog.retail_bronze.customers")
products_df = spark.table("retail_catalog.retail_bronze.products")

In [0]:
display(sales_df)

In [0]:
#Clean the data
from pyspark.sql import functions as F

sales_clean = (
    sales_df
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .filter(F.col("quantity") > 0)
    .dropDuplicates(["order_id"])
)

In [0]:
display(sales_clean)

In [0]:
#join the datasets
sales_enriched = (
    sales_clean
    .join(customers_df, "customer_id", "left")
    .join(products_df, "product_id", "left")
)

In [0]:
display(sales_enriched)

In [0]:
#Calculate sales amount
sales_enriched = sales_enriched.withColumn(
    "sales_amount",
    F.col("quantity") * F.col("unit_price")
)

In [0]:
display(sales_enriched)

In [0]:
#Write Silver
sales_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_catalog.retail_silver.sales")